In [ ]:
import pandas as pd
import torch
from torch_engine import simulate_step_gpu
import time

# 1. Wake up CUDA cores
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Engine is running on {device}")

# 2. Read in the Invincibles baseline Q matrix
epl_Q_df = pd.read_csv("./datasets/xt_Q_matrix.csv", index_col=0)

state_names = list(epl_Q_df.index)
state_to_idx = {state: i for i, state in enumerate(state_names)}
idx_to_state = {i: state for i, state in enumerate(state_names)}

Q_tensor = torch.tensor(epl_Q_df.values, dtype=torch.float32, device=device)

# 3. Create the mathematical "alive" mask
# A universe is only alive if the universe is in a P:1 state (Arsenal has possession).

p1_states = [s for s in state_names if s.startswith("P:1_")]
p1_indices = torch.tensor([state_to_idx[s] for s in p1_states], device=device)

# Get the exact integer ID for each successful goal
home_goal_idx = state_to_idx.get("HOME_GOAL", -1)

xt_grid = {}
n_simulations = 10000

print("Generating xT Grid...")
start_time = time.time()

# 4. Drop universes into every single zone
for initial_state in p1_states:
    initial_idx = state_to_idx[initial_state]

    # Spawn 10,000 universes in this specific zone
    current_states = torch.full(
        (n_simulations,), initial_idx, dtype=torch.long, device=device
    )
    # They all start alive
    active_mask = torch.ones(n_simulations, dtype=torch.bool, device=device)

    # The loop
    while active_mask.any():
        active_states = current_states[active_mask]

        # Calculate the next jump
        next_states, _ = simulate_step_gpu(active_states, Q_tensor)

        # Move the balls to the next states
        current_states[active_mask] = next_states

        # This new mask tests whether the current states [IDK NEED TO LOOK MORE INTO THIS]
        active_mask = torch.isin(current_states, p1_indices)